In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("../data/processed/desharnais.arff.csv")
df.head()

,TeamExp,ManagerExp,Transactions,Entities,PointsAdjust,Effort
0,1,4,253,52,305,5152
1,0,0,197,124,321,5635
2,4,4,40,60,100,805
3,0,0,200,119,319,3829
4,0,0,140,94,234,2149


In [4]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

(77, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   TeamExp       77 non-null     int64
 1   ManagerExp    77 non-null     int64
 2   Transactions  77 non-null     int64
 3   Entities      77 non-null     int64
 4   PointsAdjust  77 non-null     int64
 5   Effort        77 non-null     int64
dtypes: int64(6)
memory usage: 3.7 KB
None
TeamExp         0
ManagerExp      0
Transactions    0
Entities        0
PointsAdjust    0
Effort          0
dtype: int64


In [5]:
X = df.drop(columns=["Effort"])
y = df["Effort"]

In [6]:
y_log = np.log1p(y)

In [7]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

In [8]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

r2_scores = cross_val_score(xgb_model, X, y_log, cv=kf, scoring="r2")

print("R2 scores for each fold:", r2_scores)
print("Mean R2:", r2_scores.mean())

R2 scores for each fold: [ 0.19192692  0.37511884 -0.43168482 -0.08475001  0.15987338 -0.41982433
  0.1273156   0.67973314  0.40366261  0.46027015]
Mean R2: 0.14616414995061103


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42
)

rf_scores = cross_val_score(rf_model, X, y_log, cv=kf, scoring="r2")

print("RF R2 scores:", rf_scores)
print("RF Mean R2:", rf_scores.mean())

RF R2 scores: [ 0.31876272  0.39760489 -0.45859004  0.15734543  0.28681818 -0.31692322
  0.16602291  0.40487414  0.56302958  0.6114614 ]
RF Mean R2: 0.2130405996192784


In [10]:
corr = df.corr()

print(corr["Effort"].sort_values(ascending=False))

Effort          1.000000
PointsAdjust    0.703691
Transactions    0.583127
Entities        0.500223
TeamExp         0.259288
ManagerExp      0.160075
Name: Effort, dtype: float64
